![Logo Javeriana](https://images.seeklogo.com/logo-png/11/1/pontificia-universidad-javeriana-logo-png_seeklogo-110703.png)

## Nombres: 
- Gabriel Jaramillo Cuberos (Id: 20529022)
- Samuel Beltrán Martínez (Id: 20524509)
- David Vargas (Id: 20485686)
- Juan Felipe Gómez López (Id: 20553416)
- Sara Pulgarín (Id: 20491129)
- Juan Camilo Delgado (Id: 20506996)

### Primera entrega del proyecto

*Materia:* Procesamiento de Datos (1255)

*Tema:* Consultoría Nueva York.

*Descripción:* En este cuaderno hace web scraping para obtener el total de habitantes de cada uno de los distritos de la ciudad de Nueva York en el 2018.

*Nombre del fichero:* 05_Bono_Scraper.ipynb

## 1. Obtener tabla de población

Se localiza el CSV para extraer los datos.

In [1]:
import io
import re
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests


# Se descarga el archivo JavaScript que indica dónde está el CSV utilizado por la tabla de población
respuesta = requests.get(
    "https://www.baruch.cuny.edu/nycdata/Components/TableMaker.js",
    headers={"User-Agent": "Proyecto universitario NYC"},
    timeout=30,
)

# Se detiene la ejecución si la descarga falla
respuesta.raise_for_status()

# Se busca la ruta del archivo de población dentro del JavaScript
coincidencia = re.search(
    r'population:\s*"([^"]+)"',
    respuesta.text,
)

if coincidencia is None:
    raise ValueError(
        "No se encontró la dirección del CSV de población."
    )

# Se convierte la ruta encontrada en una dirección web completa
url_csv = urljoin(
    "https://www.baruch.cuny.edu/nycdata/population-geography/population.htm",
    coincidencia.group(1).strip(),
)

# Se descarga el CSV
respuesta_csv = requests.get(
    url_csv,
    headers={"User-Agent": "Proyecto universitario NYC"},
    timeout=30,
)

respuesta_csv.raise_for_status()

# Se leen las columnas como texto.
tabla = pd.read_csv(
    io.BytesIO(respuesta_csv.content),
    dtype=str,
    keep_default_na=False,
)

print("Datos descargados desde:", url_csv)

Datos descargados desde: https://www.baruch.cuny.edu/nycdata/Data/pg/population.csv


## 2. Seleccionar la ciudad de Nueva York y preparar datos del 2018

Se selecciona NYC y se preparan los datos de la población del año objetivo.

In [2]:
# Se quitan espacios sobrantes
tabla.columns = [
    re.sub(r"\s+", " ", columna).strip()
    for columna in tabla.columns
]

# Se obtienen los nombres de los territorios
tabla = tabla.rename(
    columns={tabla.columns[0]: "territorio"}
)

# Se guardan los nombres de las columnas correspondientes a los años
columnas_anios = list(tabla.columns[1:])

if "2018" not in columnas_anios:
    raise ValueError("La fuente no contiene la columna 2018.")

# Se quitan caracteres sobrantes
tabla["territorio"] = (
    tabla["territorio"]
    .str.split("^", regex=False)
    .str[0]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Se convierten las cifras a números
for columna in columnas_anios:
    tabla[columna] = pd.to_numeric(
        tabla[columna]
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce",
    )

# Se establecen encabezados
es_encabezado = tabla[columnas_anios].isna().all(axis=1)

inicio_nyc = tabla.index[
    es_encabezado & tabla["territorio"].eq("NYC")
]

if len(inicio_nyc) != 1:
    raise ValueError("No se pudo identificar una única sección NYC.")

inicio = inicio_nyc[0]

# La sección de la ciudad finaliza cuando empieza el siguiente encabezado
siguientes_encabezados = tabla.index[
    es_encabezado & (tabla.index > inicio)
]

fin = (
    siguientes_encabezados[0]
    if len(siguientes_encabezados) > 0
    else len(tabla)
)

nyc = tabla.iloc[inicio + 1:fin].copy()

# Se traducen los nombres de la fuente a los usados en el proyecto
nombres = {
    "Bronx": "Bronx",
    "Brooklyn (Kings)": "Brooklyn",
    "Manhattan (New York)": "Manhattan",
    "Queens": "Queens",
    "Staten Island (Richmond)": "Staten Island",
}

nyc["borough"] = nyc["territorio"].map(nombres)

# Se conservan los cinco distritos.
poblacion = nyc.loc[
    nyc["borough"].notna(),
    ["borough", "2018"],
].copy()

# Se verifica la extracción antes de guardar resultados
if (
    len(poblacion) != 5
    or poblacion["borough"].nunique() != 5
    or poblacion["2018"].isna().any()
    or (poblacion["2018"] <= 0).any()
):
    raise ValueError(
        "Revisar la extracción: se esperan cinco distritos "
        "diferentes con población válida."
    )

# La fuente muestra la población en miles de habitantes. Se multiplica una sola vez por 1000 para obtener habitantes
poblacion["poblacion"] = (
    poblacion["2018"] * 1000
).round().astype("int64")

poblacion["anio"] = 2018

# Se dejan las columnas necesarias para la integración.
poblacion = (
    poblacion[["anio", "borough", "poblacion"]]
    .sort_values("borough")
    .reset_index(drop=True)
)

display(poblacion)

,anio,borough,poblacion
0,2018,Bronx,1432000
1,2018,Brooklyn,2583000
2,2018,Manhattan,1629000
3,2018,Queens,2279000
4,2018,Staten Island,476000


## 3. Guardar resultados

Se guardan los resultados en la carpeta de resultados para poder ser leidos por el cuarto cuaderno.

In [3]:
carpeta_resultados = Path(
    "/opt/cluster/Proyecto/Entrega_1/resultados"
)

ruta_csv = carpeta_resultados / "poblacion_nyc_2018.csv"

poblacion.to_csv(
    ruta_csv,
    index=False,
    encoding="utf-8",
)

print("Población guardada en:", ruta_csv)
print("Unidad: habitantes, convertidos desde miles redondeados.")

Población guardada en: /opt/cluster/Proyecto/Entrega_1/resultados/poblacion_nyc_2018.csv
Unidad: habitantes, convertidos desde miles redondeados.
